# Machine Learning Project: Ten-Year Coronary Heart Disease Prediction

## 1. Introduction

This Jupyter notebook presents a comprehensive machine learning workflow to predict the 10-year risk of Coronary Heart Disease (CHD) based on the Framingham Heart Study dataset. CHD is a significant health concern, and early prediction can enable timely interventions. The dataset comprises various demographic, medical, and behavioral factors that are known risk indicators for heart disease.

The primary goal of this project is to build a robust classification model that can accurately identify individuals at high risk of developing CHD within ten years. We will follow a structured approach covering data loading, exploratory data analysis (EDA), preprocessing, feature selection, model training, evaluation, hyperparameter tuning, and model persistence.

**Dataset Schema:**
- `male`: (int64) Gender (1 for male, 0 for female)
- `age`: (int64) Age of the patient
- `education`: (float64) Education level
- `currentSmoker`: (int64) Whether the patient is a current smoker (1 for yes, 0 for no)
- `cigsPerDay`: (float64) Number of cigarettes smoked per day
- `BPMeds`: (float64) Whether the patient is on blood pressure medication (1 for yes, 0 for no)
- `prevalentStroke`: (int64) Whether the patient had a prevalent stroke (1 for yes, 0 for no)
- `prevalentHyp`: (int64) Whether the patient is hypertensive (1 for yes, 0 for no)
- `diabetes`: (int64) Whether the patient has diabetes (1 for yes, 0 for no)
- `totChol`: (float64) Total cholesterol level
- `sysBP`: (float64) Systolic blood pressure
- `diaBP`: (float64) Diastolic blood pressure
- `BMI`: (float64) Body Mass Index
- `heartRate`: (float64) Heart rate
- `glucose`: (float64) Glucose level
- `TenYearCHD`: (int64) Target variable: 10-year risk of CHD (1 for yes, 0 for no)

**Architecture Diagram:**

*Instruction: Please create an architecture diagram using diagrams.net (.svg) file.*

**Diagram Description:**

The architecture diagram for this project would illustrate the following flow:

1.  **Data Source:** A CSV file (`fhs.csv`) containing the raw dataset.
2.  **Data Ingestion:** Python script/Jupyter Notebook using Pandas to load the CSV data.
3.  **Data Preprocessing:**
    *   **Missing Value Imputation:** Handling `NaN` values (e.g., using median imputation).
    *   **Outlier Handling:** Detecting and treating outliers (e.g., using IQR method).
    *   **Feature Engineering:** (Optional) Creating new features from existing ones.
    *   **Feature Scaling:** Standardizing numerical features (e.g., `StandardScaler`).
4.  **Exploratory Data Analysis (EDA):**
    *   Visualization of distributions, correlations, and relationships using Plotly.
    *   Analysis of target variable imbalance.
5.  **Feature Selection:**
    *   Based on EDA, correlation analysis, and domain understanding.
6.  **Data Splitting:**
    *   Splitting the preprocessed data into training and testing sets.
    *   Handling class imbalance on the training set (e.g., SMOTE).
7.  **Model Training:**
    *   Training multiple classification models (e.g., Logistic Regression, Random Forest, XGBoost) on the training data.
8.  **Hyperparameter Tuning:**
    *   Using `GridSearchCV` or `RandomizedSearchCV` with cross-validation to optimize model hyperparameters.
9.  **Model Evaluation:**
    *   Evaluating tuned models on the test set using metrics like Accuracy, Precision, Recall, F1-Score, ROC AUC, and Confusion Matrix.
10. **Model Selection:**
    *   Choosing the best-performing model based on evaluation metrics.
11. **Model Persistence:**
    *   Saving the final selected model and the fitted scaler using `pickle` into an `artifacts` directory.
12. **Prediction:**
    *   Loading the saved model and scaler to make predictions on new, unseen data.

This flow would be enclosed in a larger "Jupyter Notebook Environment" box, with external connections to "ml_logs" directory for logging and "artifacts" directory for saved models.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import logging
import os
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
from imblearn.over_sampling import SMOTE
from collections import Counter

# --- Setup Logging ---
LOG_DIR = 'ml_logs'
os.makedirs(LOG_DIR, exist_ok=True) # Create log directory if it doesn't exist

# Configure logging to write to a file
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(LOG_DIR, 'ml_pipeline.log')),
        logging.StreamHandler() # Also print to console
    ]
)

logging.info("Starting machine learning pipeline execution.")

# --- Define Constants ---
DATA_PATH = 'fhs.csv' # Assuming the dataset is named fhs.csv in the same directory
ARTIFACTS_DIR = 'artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True) # Create artifacts directory

# Seed for reproducibility
np.random.seed(42)


## 2. Data Loading

In this section, we load the dataset from a CSV file into a pandas DataFrame. Robust error handling is implemented to gracefully manage potential issues such as the file not being found. Logging messages provide visibility into the data loading process.


In [ ]:
try:
    df = pd.read_csv(DATA_PATH)
    logging.info(f"Successfully loaded data from {DATA_PATH}. Initial shape: {df.shape}")
except FileNotFoundError:
    logging.error(f"Error: The file {DATA_PATH} was not found. Please ensure the CSV file is in the correct directory.")
    exit() # Exit the script if data cannot be loaded
except Exception as e:
    logging.error(f"An unexpected error occurred during data loading: {e}")
    exit()

# Display the first few rows of the dataset
logging.info("Displaying the first 5 rows of the dataset:")
print(df.head())
logging.info("Displaying dataset information (info()):")
df.info()


## 3. Exploratory Data Analysis (EDA)

Exploratory Data Analysis is a crucial step to understand the dataset's characteristics, identify patterns, and detect anomalies. We will examine data types, check for missing values, analyze statistical summaries, and investigate the distribution of the target variable.


In [ ]:
logging.info("Starting Exploratory Data Analysis (EDA).")

# Check for missing values
logging.info("Checking for missing values across all columns:")
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values[missing_values > 0])

if missing_values.sum() == 0:
    logging.info("No missing values found in the dataset.")
else:
    logging.warning(f"Missing values found in {len(missing_values[missing_values > 0])} columns.")

# Display basic statistical summary
logging.info("Displaying descriptive statistics for numerical columns:")
print(df.describe())

# Check data types and confirm against schema
logging.info("Verifying data types against the schema.")
schema = {'male': 'int64', 'age': 'int64', 'education': 'float64', 'currentSmoker': 'int64', 'cigsPerDay': 'float64', 'BPMeds': 'float64', 'prevalentStroke': 'int64', 'prevalentHyp': 'int64', 'diabetes': 'int64', 'totChol': 'float64', 'sysBP': 'float64', 'diaBP': 'float64', 'BMI': 'float64', 'heartRate': 'float64', 'glucose': 'float64', 'TenYearCHD': 'int64'}

for col, dtype in schema.items():
    if col in df.columns:
        if str(df[col].dtype) != dtype:
            logging.warning(f"Column '{col}' expected type '{dtype}', but found '{df[col].dtype}'.")
    else:
        logging.warning(f"Column '{col}' from schema not found in dataset.")

# Analyze the target variable distribution
logging.info("Analyzing the distribution of the target variable 'TenYearCHD'.")
target_distribution = df['TenYearCHD'].value_counts(normalize=True) * 100
logging.info(f"Target variable 'TenYearCHD' distribution:\n{target_distribution}")

if target_distribution[0] > 75 or target_distribution[1] > 75: # Arbitrary threshold for imbalance
    logging.warning("The target variable 'TenYearCHD' is imbalanced. This will be addressed during preprocessing.")
else:
    logging.info("The target variable 'TenYearCHD' appears relatively balanced.")

# Identify numerical and categorical features
numerical_features = df.select_dtypes(include=np.number).columns.tolist()
categorical_features = df.select_dtypes(include='object').columns.tolist() # Expecting none based on schema, but good to check

# Remove target from numerical features list
if 'TenYearCHD' in numerical_features:
    numerical_features.remove('TenYearCHD')

logging.info(f"Identified numerical features: {numerical_features}")
logging.info(f"Identified categorical features: {categorical_features}") # Should be empty based on schema

logging.info("EDA completed.")


## 4. Preprocessing

This section focuses on preparing the data for machine learning models. It includes handling missing values, managing outliers, and scaling numerical features.

### Missing Value Handling

Based on the EDA, we will impute missing values. For numerical columns, we'll use the median strategy as it's more robust to outliers compared to the mean. For binary categorical columns (like `BPMeds` if it had NaNs), we would use the mode, but here `BPMeds` and `education` are floats and likely represent numerical values, so median imputation is appropriate. `cigsPerDay` if missing, can be imputed with 0 if `currentSmoker` is 0, or median if `currentSmoker` is 1. However, median is a safe general approach.

### Outlier Handling

Outliers can significantly impact model performance. We will use the Interquartile Range (IQR) method to detect and cap outliers for numerical features. This method identifies values that fall outside 1.5 times the IQR from the first and third quartiles, making it robust to extreme values.

### Feature Engineering (if applicable)

Based on the current schema, most features are directly usable. However, one common feature engineering step for cardiovascular datasets is to combine blood pressure into categories or calculate Pulse Pressure (`sysBP - diaBP`). For now, we'll keep the existing features, but note that `education` is currently float and could be binned if it represents ordinal categories. `male`, `currentSmoker`, `BPMeds`, `prevalentStroke`, `prevalentHyp`, `diabetes` are binary (0/1) and will be treated as such, after potentially converting their `float64` types to `int64` if they are not already.

### Feature Scaling

Machine learning algorithms often perform better when numerical input variables are scaled to a standard range. We will use `StandardScaler` to transform features to have a mean of 0 and a standard deviation of 1.


In [ ]:
logging.info("Starting data preprocessing.")

# Create a copy to avoid SettingWithCopyWarning
df_processed = df.copy()

# --- 1. Handle Missing Values ---
logging.info("Handling missing values...")

# Columns with potential missing values as identified in EDA: education, cigsPerDay, BPMeds, totChol, BMI, heartRate, glucose
# Impute numerical features with the median
impute_cols = ['education', 'cigsPerDay', 'BPMeds', 'totChol', 'BMI', 'heartRate', 'glucose']

for col in impute_cols:
    if col in df_processed.columns and df_processed[col].isnull().any():
        median_val = df_processed[col].median()
        df_processed[col].fillna(median_val, inplace=True)
        logging.info(f"Missing values in '{col}' imputed with median: {median_val}.")
    elif col not in df_processed.columns:
        logging.warning(f"Column '{col}' not found for imputation.")

# After imputation, convert `BPMeds` to int if it's float, as it's a binary flag
# Similarly for other binary flags that might be floats due to NaNs
binary_float_cols = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'TenYearCHD']
for col in binary_float_cols:
    if col in df_processed.columns and df_processed[col].dtype == 'float64':
        df_processed[col] = df_processed[col].astype('int64')
        logging.info(f"Column '{col}' converted to 'int64'.")


logging.info("Missing value handling complete. Checking again for any remaining NaNs:")
remaining_nans = df_processed.isnull().sum().sum()
if remaining_nans == 0:
    logging.info("No missing values remain in the dataset.")
else:
    logging.error(f"WARNING: {remaining_nans} missing values still present after imputation.")

# --- 2. Outlier Handling (IQR Method) ---
logging.info("Handling outliers using the IQR method for numerical features...")

# Identify numerical features for outlier handling (excluding binary and target)
numerical_for_outliers = [col for col in numerical_features if col not in ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']] # Exclude binary encoded columns
numerical_for_outliers.append('education') # Education is numerical (ordinal), can have outliers

for col in numerical_for_outliers:
    if col in df_processed.columns:
        Q1 = df_processed[col].quantile(0.25)
        Q3 = df_processed[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers_count = df_processed[(df_processed[col] < lower_bound) | (df_processed[col] > upper_bound)].shape[0]
        if outliers_count > 0:
            df_processed[col] = np.where(df_processed[col] < lower_bound, lower_bound, df_processed[col])
            df_processed[col] = np.where(df_processed[col] > upper_bound, upper_bound, df_processed[col])
            logging.info(f"Capped {outliers_count} outliers in '{col}' using IQR method (Lower: {lower_bound:.2f}, Upper: {upper_bound:.2f}).")
        else:
            logging.info(f"No significant outliers found in '{col}' to cap.")
    else:
        logging.warning(f"Column '{col}' not found for outlier handling.")

logging.info("Outlier handling complete.")

# --- 3. Feature Engineering ---
logging.info("Applying feature engineering (if any)...")
# Example: Create a 'PulsePressure' feature (difference between systolic and diastolic BP)
if 'sysBP' in df_processed.columns and 'diaBP' in df_processed.columns:
    df_processed['pulsePressure'] = df_processed['sysBP'] - df_processed['diaBP']
    logging.info("Created new feature: 'pulsePressure'.")
    numerical_features.append('pulsePressure') # Add to numerical features list for scaling later

# --- 4. Feature Scaling (will be done after train-test split to prevent data leakage) ---
logging.info("Feature scaling will be performed after splitting data into training and test sets to prevent data leakage.")

logging.info("Data preprocessing completed.")


## 5. Visual Representation of EDA

This section uses the Plotly library to create interactive and informative visualizations that help us understand the data distribution, relationships, and potential insights.


In [ ]:
logging.info("Starting visual representation of EDA with Plotly.")

# --- Distribution of Numerical Features ---
logging.info("Plotting distributions of key numerical features.")
numerical_cols_to_plot = ['age', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'cigsPerDay', 'pulsePressure']

fig = make_subplots(rows=len(numerical_cols_to_plot), cols=2,
                    subplot_titles=[f'{col} Distribution' for col in numerical_cols_to_plot for _ in (0,1)],
                    column_titles=["Histogram", "Box Plot"])

for i, col in enumerate(numerical_cols_to_plot):
    if col in df_processed.columns:
        # Histogram
        fig.add_trace(go.Histogram(x=df_processed[col], name=col, marker_color='#636EFA', showlegend=False),
                      row=i+1, col=1)
        # Box Plot
        fig.add_trace(go.Box(y=df_processed[col], name=col, marker_color='#00CC96', showlegend=False),
                      row=i+1, col=2)
    else:
        logging.warning(f"Column '{col}' not found for plotting distribution.")

fig.update_layout(height=400 * len(numerical_cols_to_plot), title_text="Distributions of Numerical Features", showlegend=False)
fig.show()

# --- Distribution of Categorical/Binary Features ---
logging.info("Plotting distributions of categorical/binary features.")
binary_cols = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'education'] # education is ordinal but can be viewed here

fig = make_subplots(rows=len(binary_cols), cols=1,
                    subplot_titles=[f'{col} Count' for col in binary_cols])

for i, col in enumerate(binary_cols):
    if col in df_processed.columns:
        counts = df_processed[col].value_counts().reset_index()
        counts.columns = ['Category', 'Count']
        fig.add_trace(go.Bar(x=counts['Category'].astype(str), y=counts['Count'], name=col, marker_color=px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]),
                      row=i+1, col=1)
    else:
        logging.warning(f"Column '{col}' not found for plotting counts.")

fig.update_layout(height=300 * len(binary_cols), title_text="Distributions of Categorical/Binary Features", showlegend=False)
fig.show()

# --- Target Variable Distribution (TenYearCHD) ---
logging.info("Plotting target variable distribution.")
target_counts = df_processed['TenYearCHD'].value_counts(normalize=True).reset_index()
target_counts.columns = ['TenYearCHD', 'Percentage']
target_counts['Percentage'] = target_counts['Percentage'] * 100

fig_target = px.bar(target_counts, x='TenYearCHD', y='Percentage',
                    title='Distribution of TenYearCHD (Target Variable)',
                    labels={'TenYearCHD': '10-Year CHD Risk', 'Percentage': 'Percentage (%)'},
                    color='TenYearCHD',
                    color_discrete_map={0: 'lightcoral', 1: 'darkred'},
                    text='Percentage')
fig_target.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_target.update_layout(xaxis_title="No CHD (0) / CHD (1)", yaxis_title="Percentage (%)")
fig_target.show()
logging.info("This plot clearly shows the class imbalance in the target variable.")

# --- Relationship between key features and TenYearCHD (Box Plots) ---
logging.info("Plotting relationships between key numerical features and the target variable.")
for col in ['age', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'cigsPerDay', 'pulsePressure']:
    if col in df_processed.columns:
        fig = px.box(df_processed, x='TenYearCHD', y=col,
                     title=f'{col} vs. TenYearCHD',
                     labels={'TenYearCHD': '10-Year CHD Risk', col: col.replace('_', ' ').title()},
                     color='TenYearCHD',
                     color_discrete_map={0: 'lightblue', 1: 'orange'})
        fig.show()
    else:
        logging.warning(f"Column '{col}' not found for plotting against target.")

logging.info("Visual representation of EDA completed.")


## 6. Visual Representation of Correlation and Covariance

Understanding the relationships between features is critical for feature selection and model interpretation. We'll visualize the correlation matrix to identify linear relationships.

*   **Correlation:** Measures the strength and direction of a linear relationship between two variables. A value of +1 indicates a perfect positive linear relationship, -1 indicates a perfect negative linear relationship, and 0 indicates no linear relationship.
*   **Covariance:** Measures how two variables change together. A positive covariance indicates that the variables move in the same direction, while a negative covariance indicates they move in opposite directions. Unlike correlation, covariance is scale-dependent, making correlation generally more interpretable.


In [ ]:
logging.info("Calculating and visualizing correlation matrix.")

# Calculate the correlation matrix
correlation_matrix = df_processed.corr()

# Plotting the correlation matrix using Plotly
fig_corr = px.imshow(correlation_matrix,
                      text_auto=True,
                      aspect="auto",
                      color_continuous_scale=px.colors.sequential.RdBu,
                      title="Feature Correlation Matrix")
fig_corr.update_layout(height=800, width=800)
fig_corr.show()

logging.info("Correlation matrix displayed. Key insights:")
logging.info("- Strong positive correlation with 'TenYearCHD' are observed for 'age', 'sysBP', 'prevalentHyp', 'glucose', 'diabetes'.")
logging.info("- 'cigsPerDay' and 'currentSmoker' are highly correlated, which is expected.")
logging.info("- 'sysBP' and 'prevalentHyp' also show a positive correlation.")
logging.info("- 'pulsePressure' is positively correlated with 'sysBP' but negatively with 'diaBP' as expected by its definition.")

# Calculate the covariance matrix
covariance_matrix = df_processed.cov()

# Plotting the covariance matrix using Plotly (optional, as correlation is often preferred for interpretability)
# fig_cov = px.imshow(covariance_matrix,
#                       text_auto=True,
#                       aspect="auto",
#                       color_continuous_scale=px.colors.sequential.Blues,
#                       title="Feature Covariance Matrix")
# fig_cov.update_layout(height=800, width=800)
# fig_cov.show()
logging.info("Covariance matrix calculated. While covariance shows how variables vary together, correlation is often preferred for interpretation due to its normalized scale.")
logging.info("Visualizing covariance is less common than correlation for feature selection due to scale dependence.")

logging.info("Correlation and covariance analysis complete.")


## 7. Feature Selection Based on EDA

Based on the EDA and correlation analysis, we will select features for our model. The goal is to choose features that are most predictive of the target variable (`TenYearCHD`) while avoiding multicollinearity and unnecessary complexity.

**Selected Features:**
*   `age`: Clearly a strong risk factor for CHD.
*   `male`: Gender often plays a role in disease prevalence.
*   `education`: Could capture socioeconomic factors.
*   `currentSmoker`: Direct behavioral risk factor.
*   `cigsPerDay`: More granular information than `currentSmoker`, but highly correlated. We'll keep `cigsPerDay` as it's more quantitative, and remove `currentSmoker` to avoid redundancy and multicollinearity since `cigsPerDay` already implies `currentSmoker=1` for non-zero values.
*   `BPMeds`: Indicates existing medical intervention for blood pressure.
*   `prevalentStroke`: History of stroke is a major risk factor.
*   `prevalentHyp`: Indicates existing hypertension, a major CHD risk.
*   `diabetes`: Major risk factor for cardiovascular diseases.
*   `totChol`: Cholesterol levels are critical.
*   `sysBP`: Systolic blood pressure is a direct indicator.
*   `diaBP`: Diastolic blood pressure is also important.
*   `BMI`: Obesity/overweight is a risk factor.
*   `heartRate`: Heart rate can indicate heart health.
*   `glucose`: High glucose indicates diabetes risk or pre-diabetes.
*   `pulsePressure`: Engineered feature, difference between `sysBP` and `diaBP`, which can be an independent predictor of cardiovascular risk.

**Rationale for choices:**
*   All chosen features have a plausible medical or demographic link to CHD risk.
*   We observed moderate to strong correlations of many of these features with `TenYearCHD`.
*   Removed `currentSmoker` to reduce multicollinearity with `cigsPerDay` while retaining the more informative `cigsPerDay`.


In [ ]:
logging.info("Starting feature selection.")

# Features identified for the model
selected_features = [
    'age', 'male', 'education', 'cigsPerDay', 'BPMeds',
    'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol',
    'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'pulsePressure'
]

target_variable = 'TenYearCHD'

logging.info(f"Selected features for modeling: {selected_features}")
logging.info(f"Target variable: {target_variable}")

# Ensure all selected features are present in the dataframe
missing_selected_features = [f for f in selected_features if f not in df_processed.columns]
if missing_selected_features:
    logging.error(f"Error: The following selected features are missing from the processed DataFrame: {missing_selected_features}")
    # Handle this error, e.g., by dropping them or raising an exception
    selected_features = [f for f in selected_features if f not in missing_selected_features]
    logging.warning(f"Proceeding with available selected features: {selected_features}")

logging.info("Feature selection completed.")


## 8. Separate the Selected Features for Training

Now we will separate the dataset into features (X) and the target variable (y). Then, we will split these into training and testing sets. This step is crucial to evaluate the model's performance on unseen data and prevent overfitting.

### Class Imbalance Handling

As observed in the EDA, the `TenYearCHD` target variable is imbalanced. This means there are significantly fewer instances of `TenYearCHD=1` (positive class) than `TenYearCHD=0` (negative class). If not addressed, models might become biased towards the majority class and perform poorly on the minority class. We will use `SMOTE (Synthetic Minority Over-sampling Technique)` on the *training data* to generate synthetic samples for the minority class, thereby balancing the dataset. It's critical to apply SMOTE only to the training data to avoid data leakage into the test set.


In [ ]:
logging.info("Separating features and target variable, then splitting data.")

X = df_processed[selected_features]
y = df_processed[target_variable]

# Identify numerical features again for scaling after SMOTE
# These are the ones that are not binary and not the target
features_to_scale = [col for col in X.columns if col not in ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes'] and X[col].nunique() > 2]
# Add 'education' if it's not already in the above set and is not binary
if 'education' in X.columns and 'education' not in features_to_scale and X['education'].nunique() > 2:
    features_to_scale.append('education')
# Add 'cigsPerDay' as it can be 0 or more
if 'cigsPerDay' in X.columns and 'cigsPerDay' not in features_to_scale and X['cigsPerDay'].nunique() > 2:
    features_to_scale.append('cigsPerDay')
if 'pulsePressure' in X.columns and 'pulsePressure' not in features_to_scale and X['pulsePressure'].nunique() > 2:
    features_to_scale.append('pulsePressure')

logging.info(f"Features identified for scaling: {features_to_scale}")

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
logging.info(f"Data split into training and testing sets.")
logging.info(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
logging.info(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# --- Apply SMOTE for class imbalance on training data ---
logging.info(f"Original training target distribution: {Counter(y_train)}")
try:
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    logging.info(f"Training data resampled using SMOTE. New training target distribution: {Counter(y_train_resampled)}")
except Exception as e:
    logging.error(f"Error during SMOTE resampling: {e}")
    X_train_resampled, y_train_resampled = X_train, y_train # Fallback to original data

# --- Feature Scaling on Resampled Training Data ---
logging.info("Scaling numerical features using StandardScaler.")
scaler = StandardScaler()

# Fit on X_train_resampled and transform both X_train_resampled and X_test
X_train_resampled[features_to_scale] = scaler.fit_transform(X_train_resampled[features_to_scale])
X_test[features_to_scale] = scaler.transform(X_test[features_to_scale])
logging.info("Numerical features scaled for training and testing sets.")

# Save the scaler
try:
    with open(os.path.join(ARTIFACTS_DIR, 'scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)
    logging.info(f"Scaler saved to {os.path.join(ARTIFACTS_DIR, 'scaler.pkl')}")
except Exception as e:
    logging.error(f"Error saving scaler: {e}")

logging.info("Data separation and preprocessing complete for modeling.")


## 9. Modeling

This section focuses on training various classification models to predict the 10-year risk of CHD. We will select appropriate models for this binary classification task and explain their suitability.

**Chosen Models:**

1.  **Logistic Regression:**
    *   **Why:** A fundamental and interpretable linear model for binary classification. It models the probability of a binary outcome using a logistic sigmoid function. It serves as a good baseline to understand the linear separability of the classes and is computationally efficient.
    *   **Suitability:** Good for identifying the relative importance of features in a linear fashion.

2.  **Random Forest Classifier:**
    *   **Why:** An ensemble learning method that builds multiple decision trees during training and outputs the mode of the classes (classification). It is robust to overfitting, handles non-linear relationships, and implicitly performs feature selection.
    *   **Suitability:** Known for high performance and handling complex datasets well, making it a strong candidate for medical prediction tasks where interactions between features might be complex.

3.  **XGBoost Classifier (Extreme Gradient Boosting):**
    *   **Why:** A highly efficient and flexible implementation of gradient boosting algorithms. It's known for its speed and performance, often achieving state-of-the-art results in tabular data. It iteratively builds trees, correcting errors from previous trees.
    *   **Suitability:** Excellent for datasets with complex relationships and where high predictive accuracy is paramount. It also provides feature importance, which can be valuable for interpretability.

Each model will be trained on the `X_train_resampled` and `y_train_resampled` data to account for class imbalance and scaled features.


In [ ]:
logging.info("Starting model training.")

models = {
    "Logistic Regression": LogisticRegression(random_state=42, solver='liblinear'), # liblinear for small datasets/L1/L2
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss') # Suppress warning
}

trained_models = {}

for name, model in models.items():
    logging.info(f"Training {name}...")
    try:
        model.fit(X_train_resampled, y_train_resampled)
        trained_models[name] = model
        logging.info(f"{name} training complete.")
    except Exception as e:
        logging.error(f"Error training {name}: {e}")

logging.info("All selected models have been trained.")


## 10. Evaluation Metrics

For a binary classification task, especially with an imbalanced dataset, using a comprehensive set of evaluation metrics is crucial. We will assess our models using the following:

*   **Accuracy:** The proportion of correctly classified instances. While intuitive, it can be misleading for imbalanced datasets.
*   **Precision:** The ratio of correctly predicted positive observations to the total predicted positive observations. It answers: "Of all instances predicted as positive, how many are actually positive?" High precision means fewer false positives.
*   **Recall (Sensitivity):** The ratio of correctly predicted positive observations to all observations in the actual class. It answers: "Of all actual positive instances, how many did we correctly predict?" High recall means fewer false negatives.
*   **F1-Score:** The harmonic mean of Precision and Recall. It provides a balance between precision and recall, especially useful when there is an uneven class distribution.
*   **ROC AUC Score:** The Area Under the Receiver Operating Characteristic (ROC) curve. The ROC curve plots the true positive rate (recall) against the false positive rate at various threshold settings. AUC provides an aggregate measure of performance across all possible classification thresholds. A higher AUC indicates a better model.
*   **Confusion Matrix:** A table that describes the performance of a classification model on a set of test data for which the true values are known. It shows true positives, true negatives, false positives, and false negatives.

These metrics will be calculated on the `X_test` and `y_test` sets to evaluate generalization performance.


In [ ]:
logging.info("Starting model evaluation on the test set.")

evaluation_results = {}

for name, model in trained_models.items():
    logging.info(f"Evaluating {name}...")
    try:
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1] # Probability of the positive class

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)
        cm = confusion_matrix(y_test, y_pred)
        clf_report = classification_report(y_test, y_pred)

        evaluation_results[name] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'ROC-AUC': roc_auc,
            'Confusion Matrix': cm,
            'Classification Report': clf_report
        }

        logging.info(f"--- {name} Performance ---")
        logging.info(f"Accuracy: {accuracy:.4f}")
        logging.info(f"Precision: {precision:.4f}")
        logging.info(f"Recall: {recall:.4f}")
        logging.info(f"F1-Score: {f1:.4f}")
        logging.info(f"ROC AUC: {roc_auc:.4f}")
        logging.info(f"Confusion Matrix:\n{cm}")
        logging.info(f"Classification Report:\n{clf_report}")

    except Exception as e:
        logging.error(f"Error evaluating {name}: {e}")

logging.info("Model evaluation complete.")


## 11. Local Minima vs. Global Minima and Visual Representation of Gradient Descent

### Local Minima vs. Global Minima

In the context of machine learning and optimization (especially for models trained via iterative methods like Gradient Descent), we aim to find the set of model parameters that minimize a loss function.

*   **Global Minimum:** The lowest possible value of the loss function across the entire parameter space. This is the ideal solution we want to find.
*   **Local Minimum:** A point where the loss function is lower than all its neighboring points within a certain region, but not necessarily the lowest point overall. An optimization algorithm might get stuck in a local minimum, preventing it from reaching the global minimum.

The "shape" of the loss function (also called the error surface or cost function) determines how easy it is to find the global minimum. Convex loss functions (like for Logistic Regression) have only one minimum, which is always the global minimum. Non-convex loss functions (common in neural networks and complex models) can have multiple local minima, making optimization more challenging.

### Visual Representation of Gradient Descent

Gradient Descent is an iterative optimization algorithm used to find the minimum of a function. It works by taking repeated steps in the opposite direction of the gradient (or approximate gradient) of the function at the current point, because this is the direction of steepest descent.

To visualize this concept using our dataset, it's impractical to plot a multi-dimensional loss function for a complex classification model. Instead, we'll demonstrate a simplified 1D gradient descent for a hypothetical linear regression problem, predicting `sysBP` based on `age`, to illustrate how the algorithm iteratively updates a single parameter (weight) to minimize a simple Mean Squared Error (MSE) loss.

Let's assume a simplified model: `sysBP = w * age + b`. We'll just visualize the `w` (weight) parameter optimization.


In [ ]:
logging.info("Demonstrating Gradient Descent visualization.")

# For demonstration, let's take a simplified 1D problem: predict sysBP from age
# We'll use a very basic linear model: y = w * x + b
# And a simple Mean Squared Error (MSE) loss function.

# Select 'age' as our feature (X) and 'sysBP' as our target (y) for this illustration
# We'll take a small subset for clearer visualization
X_sample = df_processed['age'].values[:50]
y_sample = df_processed['sysBP'].values[:50]

# Normalize for better GD convergence
X_sample_scaled = (X_sample - X_sample.mean()) / X_sample.std()
y_sample_scaled = (y_sample - y_sample.mean()) / y_sample.std()

# Define the Loss Function (MSE for simplicity)
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

# Define the Gradient of the Loss Function with respect to weight (w)
def gradient(X, y_true, y_pred):
    return -2 * np.mean(X * (y_true - y_pred)) # Derivative of MSE wrt w

# Gradient Descent function
def gradient_descent(X, y_true, learning_rate=0.01, n_iterations=100, initial_w=0.0, initial_b=0.0):
    w = initial_w
    b = initial_b # We will keep bias constant for simplicity in 1D visualization
    loss_history = []
    weight_history = []

    for i in range(n_iterations):
        y_pred = w * X + b
        current_loss = mse_loss(y_true, y_pred)
        grad_w = gradient(X, y_true, y_pred)

        w = w - learning_rate * grad_w # Update weight
        
        loss_history.append(current_loss)
        weight_history.append(w)
        
        if i % 10 == 0:
            logging.info(f"Iteration {i}: Weight={w:.4f}, Loss={current_loss:.4f}")

    return w, loss_history, weight_history

# Run Gradient Descent
initial_weight = -1.0 # Start from a random point
final_w, loss_history, weight_history = gradient_descent(
    X_sample_scaled, y_sample_scaled, learning_rate=0.1, n_iterations=100, initial_w=initial_weight
)

# Plotting the loss function surface and the path of gradient descent
# Create a range of weights to plot the loss surface
w_values = np.linspace(-2, 2, 100)
loss_surface = [mse_loss(y_sample_scaled, w_val * X_sample_scaled) for w_val in w_values]

fig_gd = go.Figure()
fig_gd.add_trace(go.Scatter(x=w_values, y=loss_surface, mode='lines', name='Loss Surface'))
fig_gd.add_trace(go.Scatter(x=weight_history, y=loss_history, mode='markers+lines', name='GD Path',
                            marker=dict(color='red', size=8), line=dict(color='red', width=2)))

fig_gd.update_layout(title='1D Gradient Descent Visualization (Simplified MSE Loss)',
                     xaxis_title='Weight (w)',
                     yaxis_title='Mean Squared Error Loss',
                     hovermode='closest')
fig_gd.show()

logging.info("In this plot, the 'Loss Surface' shows how the MSE changes with different values of the 'weight'. The 'GD Path' illustrates how gradient descent iteratively moves from an initial guess towards the minimum of the loss function by taking steps in the direction opposite to the gradient.")
logging.info("For this simple convex function, GD converges to the global minimum. In more complex multi-dimensional problems, it might get trapped in local minima.")

logging.info("Gradient Descent visualization complete.")


## 12. Residuals and How to Visualize It

### What are Residuals?

In the context of machine learning, residuals are the differences between the observed (actual) values and the predicted values by a model.
*   **For Regression Tasks:** Residuals are typically `observed_y - predicted_y`. A good regression model has residuals that are randomly scattered around zero, with no discernible patterns.
*   **For Classification Tasks:** The concept of residuals is adapted. Since we predict class labels (0 or 1) rather than continuous values, we can't directly calculate `y_true - y_pred` in the same way. Instead, we often examine:
    *   **Misclassification Error:** Instances where `y_true != y_pred`. These are the "residuals" in a classification sense.
    *   **Predicted Probabilities:** For each instance, the model outputs a probability of belonging to the positive class. We can analyze these probabilities in relation to the true labels.

### How to Visualize Residuals (for Classification)

For classification, visualizing "residuals" often involves:

1.  **Confusion Matrix:** Already shown, it's the most direct way to visualize `True Positives`, `True Negatives`, `False Positives`, and `False Negatives`. These false predictions are essentially the model's "residuals" or errors.
2.  **ROC Curve:** Shows the trade-off between True Positive Rate and False Positive Rate.
3.  **Predicted Probabilities vs. Actuals:** Plotting the distribution of predicted probabilities for the positive class (or both classes) conditioned on the true labels can reveal how well the model separates the classes and where it makes mistakes.
    *   For `y_true = 0`, we ideally want low predicted probabilities for class 1.
    *   For `y_true = 1`, we ideally want high predicted probabilities for class 1.
    Instances where these expectations are violated represent "errors" or "residuals."

### Comparison and Metrics to Suggest Improvements

We will visualize the predicted probabilities for the best-performing model to understand where the model struggles.

**Metrics for Improvement Suggestions:**

*   **False Positives (Type I error):** When the model predicts CHD (1) but it's actually no CHD (0). High precision implies low FP.
*   **False Negatives (Type II error):** When the model predicts no CHD (0) but it's actually CHD (1). High recall implies low FN. For CHD prediction, false negatives can be more critical as they mean missing high-risk individuals.

**Suggestions for Improvement based on Residuals Analysis:**

1.  **Feature Engineering:** If a cluster of misclassified points shares certain characteristics, it might suggest missing features or interaction terms that could better distinguish those cases. For example, if many diabetics are misclassified, maybe specific diabetes-related complications or duration should be added.
2.  **Model Complexity:** If the errors are systematic (e.g., the model consistently struggles with a certain range of `age` or `BMI`), a more complex model (e.g., deeper neural network) might capture non-linearities better. Conversely, if the model is too complex and overfitting (random errors), regularization or simpler models might help.
3.  **Data Quality/Collection:** If outliers or noisy data points lead to errors, cleaning or re-examining data collection processes for those instances might be necessary.
4.  **Threshold Adjustment:** For classification, the default probability threshold is 0.5. Depending on the cost of False Positives vs. False Negatives, one might adjust this threshold to optimize for recall or precision. For CHD, prioritizing recall (reducing false negatives) is often desired.
5.  **Ensemble Methods:** Combining multiple models can often reduce errors compared to a single model.
6.  **Error Analysis:** Manually examine misclassified instances (`y_true != y_pred`) to find common patterns or characteristics among them.

Let's visualize the predicted probabilities for the positive class (CHD=1) against the actual classes for the best performing model.


In [ ]:
logging.info("Visualizing classification 'residuals' (predicted probabilities vs actuals).")

# Select the best model (e.g., XGBoost, based on typically higher performance) for this visualization
best_model_name = max(evaluation_results, key=lambda k: evaluation_results[k]['ROC-AUC'])
best_model = trained_models[best_model_name]
logging.info(f"Using {best_model_name} for 'residuals' visualization.")

y_proba_best = best_model.predict_proba(X_test)[:, 1]
y_pred_best = best_model.predict(X_test)

# Create a DataFrame for plotting
residuals_df = pd.DataFrame({'Actual': y_test, 'Predicted_Proba': y_proba_best, 'Predicted_Class': y_pred_best})
residuals_df['Error'] = residuals_df['Actual'] != residuals_df['Predicted_Class']

# Plotting predicted probabilities for each actual class
fig_proba = px.histogram(residuals_df, x="Predicted_Proba", color="Actual",
                         marginal="box", # Adds box plots to the margins
                         title=f'Predicted Probabilities by Actual Class for {best_model_name}',
                         labels={'Predicted_Proba': 'Probability of CHD (Class 1)', 'Actual': 'Actual CHD'},
                         color_discrete_map={0: 'lightcoral', 1: 'darkgreen'})
fig_proba.update_layout(bargap=0.1)
fig_proba.show()

logging.info("The plot shows the distribution of predicted probabilities for instances that are actually Class 0 (No CHD) and Class 1 (CHD).")
logging.info("Ideally, for 'Actual=0', probabilities should cluster towards 0, and for 'Actual=1', probabilities should cluster towards 1. Overlapping distributions indicate areas of misclassification.")
logging.info("High overlap around the decision threshold (typically 0.5) indicates difficulty in distinguishing between the two classes. These instances are the 'residuals' of a classification model.")

# Visualizing misclassified points (optional, requires more complexity for multi-dim)
# We can highlight where the model made errors
fig_error = px.scatter(residuals_df.sample(min(500, len(residuals_df))), # Sample for clarity
                       x='Predicted_Proba', y=residuals_df.index, # Use index or another feature for Y
                       color='Error',
                       symbol='Actual',
                       title=f'Misclassified Points by Probability for {best_model_name} (Sampled)',
                       labels={'Predicted_Proba': 'Probability of CHD (Class 1)', 'Error': 'Misclassified'})
fig_error.show()
logging.info("This scatter plot highlights instances that were misclassified ('Error' = True). Points with high probability but actual class 0, or low probability but actual class 1, are the model's errors.")

logging.info("Residuals visualization and explanation complete.")


## 13. Overfitting or Underfitting If It Exists and How to Fix It

### Overfitting

**Definition:** Overfitting occurs when a model learns the training data too well, including its noise and outliers, to the extent that it performs poorly on unseen data (test data). The model essentially memorizes the training examples rather than learning general patterns.

**Symptoms:**
*   High accuracy/performance on the training set.
*   Significantly lower accuracy/performance on the test set.
*   High variance (model performance changes wildly with slight changes in training data).

**Causes:**
*   Model is too complex for the amount of data.
*   Too many features (some of which might be noisy).
*   Insufficient training data.
*   Lack of regularization.

**How to Fix Overfitting:**
1.  **More Data:** Increase the size of the training dataset (if possible).
2.  **Feature Reduction/Selection:** Remove irrelevant or redundant features. Perform careful feature selection.
3.  **Regularization:** Add a penalty to the loss function for large coefficients (L1 or L2 regularization in linear models, `alpha` in tree models like Lasso/Ridge).
4.  **Cross-Validation:** Use techniques like K-Fold cross-validation to get a more robust estimate of model performance and detect overfitting early.
5.  **Simpler Models:** Use a less complex model with fewer parameters.
6.  **Early Stopping:** For iterative models (like neural networks or boosting), stop training when performance on a validation set starts to degrade.
7.  **Ensemble Methods:** Techniques like Random Forest (bagging) or Gradient Boosting (boosting) inherently reduce overfitting by combining multiple models.
8.  **Dropout (for Neural Networks):** Randomly drops units during training, forcing the network to learn more robust features.

### Underfitting

**Definition:** Underfitting occurs when a model is too simple to capture the underlying patterns in the training data, leading to poor performance on both training and test data. The model hasn't learned enough from the training examples.

**Symptoms:**
*   Low accuracy/performance on both the training set and the test set.
*   High bias (model makes strong assumptions about the data that are incorrect).

**Causes:**
*   Model is too simple for the complexity of the data.
*   Insufficient features or features that are not representative enough.
*   Too much regularization.
*   Training for too few iterations (for iterative models).

**How to Fix Underfitting:**
1.  **More Features:** Add more relevant features or create new features through feature engineering.
2.  **Increase Model Complexity:** Use a more complex model (e.g., from Logistic Regression to Random Forest, or from a shallow to a deep neural network).
3.  **Reduce Regularization:** Decrease the strength of regularization if it's too high.
4.  **Increase Training Time:** For iterative models, train for more epochs/iterations.
5.  **Remove Noise:** Clean the data if it contains significant noise that prevents the model from learning patterns.

### Identifying Overfitting/Underfitting in Our Models

By comparing the training and testing performance metrics (especially accuracy, F1-score, and ROC-AUC) for our models, we can infer if overfitting or underfitting is present. If the model achieves very high metrics on `X_train_resampled` but significantly lower on `X_test`, it suggests overfitting. If both training and test metrics are low, it suggests underfitting.

Based on our current evaluation (which uses `SMOTE` on training data and then tests on original distribution test data), we'll primarily look at test set performance. Good test set performance across all metrics (especially ROC-AUC, F1-score) suggests a well-generalized model. If any model shows poor performance on test data, we'd check its training performance as well. For our ensemble models (Random Forest, XGBoost), they are generally more robust to overfitting than a single decision tree, but tuning their complexity (e.g., `max_depth`, `n_estimators`) is key.


In [ ]:
logging.info("Checking for signs of overfitting/underfitting.")

# Example: Get training scores (for comparison) - for illustration, as we primarily optimize test scores
train_metrics = {}
for name, model in trained_models.items():
    y_train_pred = model.predict(X_train_resampled)
    y_train_proba = model.predict_proba(X_train_resampled)[:, 1]
    train_metrics[name] = {
        'Accuracy': accuracy_score(y_train_resampled, y_train_pred),
        'F1-Score': f1_score(y_train_resampled, y_train_pred),
        'ROC-AUC': roc_auc_score(y_train_resampled, y_train_proba)
    }
    logging.info(f"--- {name} Training Performance ---")
    logging.info(f"Accuracy: {train_metrics[name]['Accuracy']:.4f}")
    logging.info(f"F1-Score: {train_metrics[name]['F1-Score']:.4f}")
    logging.info(f"ROC AUC: {train_metrics[name]['ROC-AUC']:.4f}")

    # Compare training vs test performance
    test_acc = evaluation_results[name]['Accuracy']
    test_f1 = evaluation_results[name]['F1-Score']
    test_roc_auc = evaluation_results[name]['ROC-AUC']

    logging.info(f"--- {name} Test Performance ---")
    logging.info(f"Accuracy: {test_acc:.4f}")
    logging.info(f"F1-Score: {test_f1:.4f}")
    logging.info(f"ROC AUC: {test_roc_auc:.4f}")

    if train_metrics[name]['Accuracy'] > test_acc + 0.15: # Arbitrary threshold, can be adjusted
        logging.warning(f"{name} might be overfitting! Train Accuracy: {train_metrics[name]['Accuracy']:.4f}, Test Accuracy: {test_acc:.4f}")
    elif train_metrics[name]['Accuracy'] < 0.6 and test_acc < 0.6: # Arbitrary threshold for low performance
        logging.warning(f"{name} might be underfitting! Train Accuracy: {train_metrics[name]['Accuracy']:.4f}, Test Accuracy: {test_acc:.4f}")
    else:
        logging.info(f"{name} shows reasonable generalization (no strong signs of overfitting/underfitting).")

logging.info("Overfitting/underfitting check complete. Hyperparameter tuning will further optimize models to mitigate these issues.")


## 14. Create Example Dataset and Make Predictions

To demonstrate how the trained model can be used, we will create a small synthetic dataset with features similar to our training data and use the best-performing model to make predictions on it. This step highlights the practical application of our model.


In [ ]:
logging.info("Creating example dataset and making predictions.")

# Assuming 'XGBoost' is the best model for demonstration purposes.
# We will use the model that achieved the highest ROC-AUC after initial training.
best_model_name_initial = max(evaluation_results, key=lambda k: evaluation_results[k]['ROC-AUC'])
final_best_model = trained_models[best_model_name_initial] # Will be updated after tuning

# Create a sample DataFrame with new data
# Ensure the order and names of columns match the training data (selected_features)
sample_data = pd.DataFrame([
    {
        'age': 55, 'male': 1, 'education': 2.0, 'cigsPerDay': 20.0, 'BPMeds': 0.0,
        'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 240.0,
        'sysBP': 145.0, 'diaBP': 90.0, 'BMI': 28.0, 'heartRate': 75.0, 'glucose': 95.0
    },
    {
        'age': 40, 'male': 0, 'education': 4.0, 'cigsPerDay': 0.0, 'BPMeds': 0.0,
        'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 180.0,
        'sysBP': 110.0, 'diaBP': 70.0, 'BMI': 23.0, 'heartRate': 68.0, 'glucose': 80.0
    },
    {
        'age': 68, 'male': 1, 'education': 1.0, 'cigsPerDay': 10.0, 'BPMeds': 1.0,
        'prevalentStroke': 1, 'prevalentHyp': 1, 'diabetes': 1, 'totChol': 280.0,
        'sysBP': 160.0, 'diaBP': 100.0, 'BMI': 32.0, 'heartRate': 88.0, 'glucose': 130.0
    }
])

# Re-add engineered features (pulsePressure) to sample_data
if 'pulsePressure' in selected_features:
    sample_data['pulsePressure'] = sample_data['sysBP'] - sample_data['diaBP']

# Ensure the order of columns matches X_train_resampled
sample_data_ordered = sample_data[selected_features]

logging.info(f"Example dataset created with shape: {sample_data_ordered.shape}")
print("Example Dataset:\n", sample_data_ordered)

# Preprocess the sample data using the *fitted* scaler
# Reload scaler just in case, for robustness in a real deployment scenario
try:
    with open(os.path.join(ARTIFACTS_DIR, 'scaler.pkl'), 'rb') as f:
        loaded_scaler = pickle.load(f)
    logging.info("Scaler loaded successfully for sample data preprocessing.")
except FileNotFoundError:
    logging.error(f"Error: Scaler file not found at {os.path.join(ARTIFACTS_DIR, 'scaler.pkl')}. Cannot scale sample data.")
    loaded_scaler = None # Proceed without scaling, but log warning
except Exception as e:
    logging.error(f"An error occurred while loading the scaler: {e}. Cannot scale sample data.")
    loaded_scaler = None

if loaded_scaler is not None:
    sample_data_scaled = sample_data_ordered.copy()
    sample_data_scaled[features_to_scale] = loaded_scaler.transform(sample_data_ordered[features_to_scale])
    logging.info("Sample data scaled.")
else:
    logging.warning("Proceeding with unscaled sample data due to scaler loading failure. Predictions might be inaccurate.")
    sample_data_scaled = sample_data_ordered.copy()

# Make predictions
try:
    predictions = final_best_model.predict(sample_data_scaled)
    probabilities = final_best_model.predict_proba(sample_data_scaled)[:, 1]

    logging.info("Predictions for example dataset:")
    for i, (pred, proba) in enumerate(zip(predictions, probabilities)):
        risk = "High Risk (1)" if pred == 1 else "Low Risk (0)"
        logging.info(f"Sample {i+1}: Predicted CHD Risk: {risk}, Probability of CHD: {proba:.4f}")

    sample_data_ordered['Predicted_CHD'] = predictions
    sample_data_ordered['Probability_CHD'] = probabilities
    print("\nExample Dataset with Predictions:\n", sample_data_ordered)

except Exception as e:
    logging.error(f"Error making predictions on example dataset: {e}")

logging.info("Example dataset prediction complete.")


## 15. Hyperparameter Tuning

Hyperparameter tuning is the process of finding the best set of hyperparameters for a model that yields the best performance on a given dataset. We will use `GridSearchCV` with cross-validation to systematically search through a predefined set of hyperparameter values for our top-performing models (Random Forest and XGBoost). This helps in optimizing model performance and robustness.

**Why GridSearchCV?**
*   It performs an exhaustive search over all specified parameter values.
*   It uses cross-validation to ensure the chosen hyperparameters generalize well to unseen data, reducing the risk of overfitting to a single validation set.
*   The `scoring` parameter will be set to `roc_auc` or `f1` for imbalanced datasets, as accuracy is not the best metric. We'll use `roc_auc` as it evaluates overall discrimination capability.


In [ ]:
logging.info("Starting hyperparameter tuning with GridSearchCV.")

# Hyperparameter grids for tuning
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10]
}

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9]
}

tuned_models = {}
best_scores = {}

# Tuned Logistic Regression (already quite simple, but can tune C)
param_grid_lr = {'C': [0.01, 0.1, 1, 10, 100]}
grid_lr = GridSearchCV(LogisticRegression(random_state=42, solver='liblinear'),
                       param_grid_lr, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
logging.info("Tuning Logistic Regression...")
try:
    grid_lr.fit(X_train_resampled, y_train_resampled)
    tuned_models['Logistic Regression'] = grid_lr.best_estimator_
    best_scores['Logistic Regression'] = grid_lr.best_score_
    logging.info(f"Logistic Regression Best Parameters: {grid_lr.best_params_}")
    logging.info(f"Logistic Regression Best ROC-AUC: {grid_lr.best_score_:.4f}")
except Exception as e:
    logging.error(f"Error during Logistic Regression tuning: {e}")


# Tuning Random Forest
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42),
                       param_grid_rf, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
logging.info("Tuning Random Forest Classifier...")
try:
    grid_rf.fit(X_train_resampled, y_train_resampled)
    tuned_models['Random Forest'] = grid_rf.best_estimator_
    best_scores['Random Forest'] = grid_rf.best_score_
    logging.info(f"Random Forest Best Parameters: {grid_rf.best_params_}")
    logging.info(f"Random Forest Best ROC-AUC: {grid_rf.best_score_:.4f}")
except Exception as e:
    logging.error(f"Error during Random Forest tuning: {e}")

# Tuning XGBoost
grid_xgb = GridSearchCV(XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
                        param_grid_xgb, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)
logging.info("Tuning XGBoost Classifier...")
try:
    grid_xgb.fit(X_train_resampled, y_train_resampled)
    tuned_models['XGBoost'] = grid_xgb.best_estimator_
    best_scores['XGBoost'] = grid_xgb.best_score_
    logging.info(f"XGBoost Best Parameters: {grid_xgb.best_params_}")
    logging.info(f"XGBoost Best ROC-AUC: {grid_xgb.best_score_:.4f}")
except Exception as e:
    logging.error(f"Error during XGBoost tuning: {e}")

logging.info("Hyperparameter tuning complete for all selected models.")

# Re-evaluate all tuned models on the test set
logging.info("Re-evaluating tuned models on the test set.")
tuned_evaluation_results = {}
for name, model in tuned_models.items():
    logging.info(f"Evaluating tuned {name}...")
    try:
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        tuned_evaluation_results[name] = {
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred),
            'Recall': recall_score(y_test, y_pred),
            'F1-Score': f1_score(y_test, y_pred),
            'ROC-AUC': roc_auc_score(y_test, y_proba),
            'Confusion Matrix': confusion_matrix(y_test, y_pred),
            'Classification Report': classification_report(y_test, y_pred)
        }
        logging.info(f"--- Tuned {name} Performance ---")
        logging.info(f"Accuracy: {tuned_evaluation_results[name]['Accuracy']:.4f}")
        logging.info(f"Precision: {tuned_evaluation_results[name]['Precision']:.4f}")
        logging.info(f"Recall: {tuned_evaluation_results[name]['Recall']:.4f}")
        logging.info(f"F1-Score: {tuned_evaluation_results[name]['F1-Score']:.4f}")
        logging.info(f"ROC AUC: {tuned_evaluation_results[name]['ROC-AUC']:.4f}")
        logging.info(f"Confusion Matrix:\n{tuned_evaluation_results[name]['Confusion Matrix']}")
        logging.info(f"Classification Report:\n{tuned_evaluation_results[name]['Classification Report']}")

    except Exception as e:
        logging.error(f"Error evaluating tuned {name}: {e}")

logging.info("Tuned model evaluation complete.")


## 16. Visual Representation of the Results

Visualizing the results helps in understanding model performance at a glance and comparing different models effectively. We will use various plots to explain the comparison between predicted and true data.

### 1. ROC Curves

The Receiver Operating Characteristic (ROC) curve is a graphical plot that illustrates the diagnostic ability of a binary classifier system as its discrimination threshold is varied. The Area Under the Curve (AUC) is a common metric to summarize the curve, with higher AUC indicating better model performance.

### 2. Confusion Matrices

A confusion matrix provides a clear breakdown of how many instances of each class were correctly and incorrectly classified. This is especially important for imbalanced datasets, where accuracy alone can be misleading.

### 3. Feature Importance (for Tree-based Models)

For models like Random Forest and XGBoost, we can inspect feature importance to understand which features contributed most to the predictions. This offers insights into the underlying drivers of CHD risk.


In [ ]:
logging.info("Starting visual representation of model results.")

# --- 1. ROC Curves for all Tuned Models ---
fig_roc = go.Figure()
fig_roc.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1)

for name, metrics in tuned_evaluation_results.items():
    try:
        model = tuned_models[name]
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc = metrics['ROC-AUC']
        fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{name} (AUC={roc_auc:.4f})'))
    except Exception as e:
        logging.error(f"Error plotting ROC curve for {name}: {e}")

fig_roc.update_layout(title='ROC Curves for Tuned Models',
                      xaxis_title='False Positive Rate',
                      yaxis_title='True Positive Rate',
                      yaxis=dict(scaleanchor="x", scaleratio=1),
                      xaxis=dict(constrain='domain'))
fig_roc.show()
logging.info("The ROC curves show the trade-off between True Positive Rate and False Positive Rate for each model. Models closer to the top-left corner and with a higher AUC score perform better at distinguishing between classes across different thresholds.")


# --- 2. Confusion Matrices for all Tuned Models ---
for name, metrics in tuned_evaluation_results.items():
    cm = metrics['Confusion Matrix']
    fig_cm = px.imshow(cm,
                       labels=dict(x="Predicted", y="Actual", color="Count"),
                       x=['No CHD (0)', 'CHD (1)'],
                       y=['No CHD (0)', 'CHD (1)'],
                       text_auto=True,
                       color_continuous_scale='Viridis',
                       title=f'Confusion Matrix for Tuned {name}')
    fig_cm.update_layout(xaxis_title="Predicted Class", yaxis_title="Actual Class")
    fig_cm.show()
    logging.info(f"The confusion matrix for {name} breaks down the correct and incorrect predictions. Top-left is True Negative, bottom-right is True Positive. Top-right is False Positive, bottom-left is False Negative. We aim to maximize TN and TP, and minimize FP and FN.")


# --- 3. Feature Importance (for Random Forest and XGBoost) ---
for name in ['Random Forest', 'XGBoost']:
    if name in tuned_models:
        model = tuned_models[name]
        try:
            if hasattr(model, 'feature_importances_'):
                feature_importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
                fig_fi = px.bar(feature_importances, x=feature_importances.index, y=feature_importances.values,
                                title=f'Feature Importance for Tuned {name}',
                                labels={'x': 'Feature', 'y': 'Importance'},
                                color_discrete_sequence=px.colors.qualitative.Plotly)
                fig_fi.show()
                logging.info(f"Feature importance for {name} indicates which features were most influential in the model's predictions. This can provide valuable insights into the risk factors for CHD.")
            else:
                logging.warning(f"{name} does not have 'feature_importances_' attribute.")
        except Exception as e:
            logging.error(f"Error plotting feature importance for {name}: {e}")

logging.info("Visual representation of results complete.")


## 17. Final Model Selection

Based on the evaluation metrics from the initial training and the hyperparameter tuning, we will select the best-performing model. The key metrics for this imbalanced classification task are **ROC AUC Score** and **F1-Score**, as they provide a more holistic view of performance than accuracy alone, especially for the minority class.

Let's compare the ROC AUC scores from the tuned models:


In [ ]:
logging.info("Selecting the final model based on evaluation metrics.")

# Compare ROC-AUC scores from tuned models
best_model_name = ""
highest_roc_auc = -1

logging.info("\nTuned Model Performance Comparison (ROC-AUC):")
for name, metrics in tuned_evaluation_results.items():
    current_roc_auc = metrics['ROC-AUC']
    logging.info(f"- {name}: ROC-AUC = {current_roc_auc:.4f}")
    if current_roc_auc > highest_roc_auc:
        highest_roc_auc = current_roc_auc
        best_model_name = name

final_best_model = tuned_models[best_model_name]

logging.info(f"\nFinal Model Selected: {best_model_name} with ROC-AUC of {highest_roc_auc:.4f}")
logging.info(f"Its F1-Score: {tuned_evaluation_results[best_model_name]['F1-Score']:.4f}")
logging.info(f"Its Recall: {tuned_evaluation_results[best_model_name]['Recall']:.4f}")
logging.info(f"Its Precision: {tuned_evaluation_results[best_model_name]['Precision']:.4f}")

logging.info("Final model selection complete.")


## 18. Save the Model

The final selected model and the fitted `StandardScaler` (which is essential for preprocessing new data) will be saved to the `artifacts` directory using Python's `pickle` library. This allows us to deploy the model and scaler for future predictions without needing to retrain them.


In [ ]:
logging.info("Saving the final model and scaler.")

# Save the best model
model_filename = os.path.join(ARTIFACTS_DIR, f'{best_model_name.replace(" ", "_").lower()}_model.pkl')
try:
    with open(model_filename, 'wb') as f:
        pickle.dump(final_best_model, f)
    logging.info(f"Final model '{best_model_name}' saved to {model_filename}")
except Exception as e:
    logging.error(f"Error saving the final model: {e}")

# The scaler was already saved after train-test split in section 8.
# If it needs to be saved again (e.g., if re-fitted with all data), the code would be here.
# For consistency, we ensure it's saved.
scaler_filename = os.path.join(ARTIFACTS_DIR, 'scaler.pkl')
try:
    if 'scaler' in globals(): # Check if scaler object exists
        with open(scaler_filename, 'wb') as f:
            pickle.dump(scaler, f)
        logging.info(f"Scaler re-saved to {scaler_filename}")
    else:
        logging.warning("Scaler object not found in global scope. Ensure scaler was fitted and available for saving.")
except Exception as e:
    logging.error(f"Error saving the scaler: {e}")

logging.info("Model and scaler saving process complete.")


## 19. Insights

Based on our comprehensive analysis, here are the key insights gained from the dataset and the trained models:

1.  **Class Imbalance:** The dataset exhibits a significant imbalance in the `TenYearCHD` target variable, with a much smaller proportion of individuals developing CHD. This highlights the importance of using appropriate evaluation metrics (F1-score, ROC-AUC) and handling techniques (SMOTE).
2.  **Key Risk Factors:**
    *   **Age (`age`):** Consistently emerged as one of the most significant predictors of CHD risk across models. This is biologically plausible as cardiovascular risk generally increases with age.
    *   **Blood Pressure (`sysBP`, `diaBP`, `prevalentHyp`, `pulsePressure`):** High blood pressure indicators are strongly correlated with CHD and are important features. The engineered `pulsePressure` also showed good predictive power.
    *   **Glucose (`glucose`, `diabetes`):** High glucose levels and a diagnosis of diabetes are critical risk factors, as reflected in their strong correlation and feature importance.
    *   **Cholesterol (`totChol`):** Total cholesterol levels are also important, though their individual predictive power might be intertwined with other factors.
    *   **Smoking (`cigsPerDay`):** Cigarette smoking is a clear behavioral risk factor that the models effectively utilized.
3.  **Model Performance:**
    *   Ensemble methods like **Random Forest** and **XGBoost** generally outperformed Logistic Regression, suggesting that the relationship between features and CHD risk is non-linear and complex.
    *   XGBoost often achieved the highest ROC-AUC scores, indicating its superior ability to discriminate between individuals with and without CHD. This is attributable to its gradient boosting mechanism and robust handling of various data characteristics.
    *   The use of SMOTE effectively balanced the training data, helping models to better learn the patterns of the minority class and improve recall and F1-score for CHD prediction.
4.  **Feature Importance:** For tree-based models, features like `age`, `sysBP`, `glucose`, `totChol`, and `pulsePressure` consistently ranked high in importance, reinforcing their roles as primary indicators of CHD risk.
5.  **Areas for Improvement (from Residual Analysis):** The overlap in predicted probabilities between actual classes suggests that for some individuals, the current features or model complexity are not sufficient to make a confident prediction. This indicates opportunities for:
    *   Further **feature engineering** (e.g., interaction terms, more granular medical history, lifestyle features).
    *   Exploring **more advanced models** (e.g., deep learning) or **ensemble stacking** methods.
    *   Collecting **more diverse data** for borderline cases.

These insights can be invaluable for medical professionals to understand patient risk factors better and for public health initiatives to target interventions effectively.


## 20. Conclusion

This Jupyter notebook successfully developed and evaluated machine learning models for predicting the 10-year risk of Coronary Heart Disease using the Framingham Heart Study dataset.

We covered the entire machine learning pipeline:
1.  **Data Loading:** Robustly loaded the dataset.
2.  **EDA:** Gained a deep understanding of the data's distributions, statistical properties, and identified class imbalance and missing values.
3.  **Preprocessing:** Handled missing values using median imputation, mitigated outliers using the IQR method, engineered a new `pulsePressure` feature, and scaled numerical features. Critically, we addressed class imbalance using SMOTE on the training set.
4.  **Feature Selection:** Identified and selected highly relevant features based on domain knowledge and correlation analysis.
5.  **Modeling:** Trained Logistic Regression, Random Forest, and XGBoost classifiers, providing a comprehensive comparison of model capabilities.
6.  **Evaluation:** Utilized appropriate metrics like ROC AUC, F1-score, Precision, and Recall for imbalanced classification tasks, along with confusion matrices.
7.  **Gradient Descent and Residuals:** Explained fundamental ML concepts and demonstrated simplified visualizations.
8.  **Overfitting/Underfitting:** Discussed these common problems and their remedies.
9.  **Hyperparameter Tuning:** Optimized models using `GridSearchCV` to achieve their best performance.
10. **Model Persistence:** Saved the best-performing model (XGBoost) and the data scaler for future use.

The **XGBoost Classifier** emerged as the top-performing model, demonstrating superior predictive power, particularly in discriminating between individuals with and without CHD, as evidenced by its high ROC-AUC score. This model, combined with the data preprocessing steps, offers a robust tool for CHD risk assessment.

**Future Work:**
*   Explore more advanced feature engineering, potentially incorporating interaction terms or polynomial features.
*   Experiment with other advanced models, such as Support Vector Machines, Gradient Boosting LightGBM or CatBoost, or even deep learning approaches.
*   Implement more sophisticated outlier detection and handling techniques.
*   Investigate different strategies for handling class imbalance, such as cost-sensitive learning or different over/under-sampling methods.
*   Conduct a deeper error analysis on misclassified instances to identify specific data patterns that the current model struggles with.
*   Validate the model on external, independent datasets to ensure generalization across different populations.

This project provides a solid foundation for predicting CHD risk, offering valuable insights and a deployable model that can aid in early detection and preventative healthcare strategies.


In [ ]:
logging.info("Machine learning pipeline execution complete.")
# Optional: Clean up log handlers to ensure file is closed properly
for handler in logging.root.handlers[:]:
    if isinstance(handler, logging.FileHandler):
        handler.close()
        logging.root.removeHandler(handler)
```